In [6]:
import pandas as pd

# Load all sheets as a dictionary
sheets = pd.read_excel(
    r"C:\Users\jumma\Downloads\online+retail+ii\online_retail_II.xlsx",
    sheet_name=None
)

# Check what sheets you got
print(sheets.keys())

# Combine into one DataFrame
df = pd.concat(sheets.values(), ignore_index=True)

print(df.shape)
df.info()
df.isnull().sum()

dict_keys(['Year 2009-2010', 'Year 2010-2011'])
(1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [10]:
df.head()
df.shape           # ~1M+ rows expected across both sheets
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [7]:
df['InvoiceDate'].min(), df['InvoiceDate'].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:50:00'))

In [12]:
print("Duplicates:", df.duplicated().sum())
print("\nQuantity stats:")
print(df['Quantity'].describe())
print("\nPrice stats:")
print(df['Price'].describe())
print("\nCancellations:", df['Invoice'].astype(str).str.startswith('C').sum())
print("\nTop 10 countries:")
print(df['Country'].value_counts().head(10))

Duplicates: 34335

Quantity stats:
count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

Price stats:
count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64

Cancellations: 19494

Top 10 countries:
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64


In [13]:
# Negative prices
print("Negative price rows:")
print(df[df['Price'] < 0][['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head(10))

# Extreme quantities
print("\nExtreme quantity rows (|qty| > 10000):")
print(df[df['Quantity'].abs() > 10000][['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']])

# Non-product StockCodes (these cause most outliers)
print("\nSuspicious StockCodes:")
print(df[df['StockCode'].astype(str).str.len() <= 3]['StockCode'].value_counts().head(20))

Negative price rows:
        Invoice StockCode      Description  Quantity     Price
179403  A506401         B  Adjust bad debt         1 -53594.36
276274  A516228         B  Adjust bad debt         1 -44031.79
403472  A528059         B  Adjust bad debt         1 -38925.87
825444  A563186         B  Adjust bad debt         1 -11062.06
825445  A563187         B  Adjust bad debt         1 -11062.06

Extreme quantity rows (|qty| > 10000):
         Invoice StockCode                         Description  Quantity  \
90857     497946     37410  BLACK AND WHITE PAISLEY FLOWER MUG     19152   
127166    501534     21099         SET/6 STRAWBERRY PAPER CUPS     12960   
127167    501534     21092       SET/6 STRAWBERRY PAPER PLATES     12480   
127168    501534     21091         SET/6 WOODLAND PAPER PLATES     12960   
127169    501534     21085           SET/6 WOODLAND PAPER CUPS     12744   
192197    507637     84016          FLAG OF ST GEORGE CAR FLAG     10200   
587080    541431     23166   

In [14]:
# === STEP 1: Drop exact duplicates ===
before = len(df)
df = df.drop_duplicates()
print(f"✓ Removed {before - len(df):,} duplicates")

# === STEP 2: Drop missing descriptions ===
before = len(df)
df = df.dropna(subset=['Description'])
df['Description'] = df['Description'].str.strip().str.upper()
print(f"✓ Removed {before - len(df):,} rows with missing Description")

# === STEP 3: Flag customer type, fill missing IDs ===
df['CustomerType'] = df['Customer ID'].apply(
    lambda x: 'Guest' if pd.isna(x) else 'Registered'
)
df['Customer ID'] = df['Customer ID'].fillna(0).astype(int)
print(f"✓ Flagged {(df['CustomerType']=='Guest').sum():,} guest transactions")

# === STEP 4: Flag cancellations ===
df['IsCancellation'] = df['Invoice'].astype(str).str.startswith('C')
print(f"✓ Flagged {df['IsCancellation'].sum():,} cancellations")

# === STEP 5: Remove non-product admin entries ===
admin_codes = ['DOT', 'M', 'C2', 'D', 'S', 'B', 'POST', 'BANK CHARGES', 
               'AMAZONFEE', 'CRUK', 'PADS', 'TEST001', 'TEST002', 'GIFT',
               'ADJUST', 'ADJUST2', 'SP1002', 'm', 'C3']
before = len(df)
df = df[~df['StockCode'].astype(str).isin(admin_codes)]
print(f"✓ Removed {before - len(df):,} admin/non-product rows")

# === STEP 6: Remove invalid prices and quantities ===
before = len(df)
df = df[df['Price'] > 0]
df = df[df['Quantity'] != 0]
print(f"✓ Removed {before - len(df):,} rows with invalid price/quantity")

# === STEP 7: Clean country names ===
df['Country'] = df['Country'].str.strip()

# === STEP 8: Add derived columns ===
df['Revenue'] = df['Quantity'] * df['Price']
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['MonthName'] = df['InvoiceDate'].dt.month_name()
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
print(f"✓ Added derived columns")

# === STEP 9: Final summary ===
print(f"\n{'='*50}")
print(f"FINAL DATASET")
print(f"{'='*50}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Total revenue: £{df['Revenue'].sum():,.2f}")
print(f"Unique customers: {df[df['CustomerType']=='Registered']['Customer ID'].nunique():,}")
print(f"Unique products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique()}")
print(f"\nNulls remaining:")
print(df.isnull().sum())

✓ Removed 34,335 duplicates
✓ Removed 4,275 rows with missing Description
✓ Flagged 230,876 guest transactions
✓ Flagged 19,104 cancellations
✓ Removed 5,707 admin/non-product rows
✓ Removed 1,724 rows with invalid price/quantity
✓ Added derived columns

FINAL DATASET
Rows: 1,021,330
Columns: 17
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Total revenue: £18,929,107.51
Unique customers: 5,875
Unique products: 4,915
Countries: 43

Nulls remaining:
Invoice           0
StockCode         0
Description       0
Quantity          0
InvoiceDate       0
Price             0
Customer ID       0
Country           0
CustomerType      0
IsCancellation    0
Revenue           0
Year              0
Month             0
MonthName         0
DayOfWeek         0
Hour              0
YearMonth         0
dtype: int64


In [16]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,CustomerType,IsCancellation,Revenue,Year,Month,MonthName,DayOfWeek,Hour,YearMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Registered,False,83.4,2009,12,December,Tuesday,7,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Registered,False,81.0,2009,12,December,Tuesday,7,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Registered,False,81.0,2009,12,December,Tuesday,7,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Registered,False,100.8,2009,12,December,Tuesday,7,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Registered,False,30.0,2009,12,December,Tuesday,7,2009-12


In [18]:
df['Invoice'] = df['Invoice'].astype(str)

In [19]:
df['Invoice'] = df['Invoice'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

In [20]:
# Force text columns to be strings
df['Invoice'] = df['Invoice'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

# Save CSV first (always works)
df.to_csv(r"C:\Users\jumma\Downloads\online_retail_cleaned.csv", index=False)
print("✓ CSV saved")

# Then try Parquet
df.to_parquet(r"C:\Users\jumma\Downloads\online_retail_cleaned.parquet")
print("✓ Parquet saved")

✓ CSV saved
✓ Parquet saved
